# MvDEC S2, 40 seeds (52-91) + label trajectory - REUTERS

Controlled diagnostic only (`controlled_diagnostic_not_exact_paper_truth`). Same S2 protocol as
`mvdec_2025_scratch_both_views_min500_reuters_seeds52_91_kaggle.ipynb`, plus a label-aware
per-checkpoint trajectory written to a sidecar (`MVDEC_LABEL_TRAJECTORY_LOG`). The trajectory
never feeds back into training, so manifests and config hashes are identical to the plain run.

It tests two hypotheses for the Table 2 "best performance" row together:
H1 best-of-N seeds selected by labels, H2 best intermediate checkpoint (DEKM's release code
prints ACC/NMI at every K-Means refresh). Both are label-guided and are reported only to see
whether the paper numbers are reachable that way.

Kaggle settings: GPU on, Internet on, no input dataset. About 40 x 6-10 GPU minutes.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a Kaggle GPU before running this diagnostic."

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

PROTOCOL = "mvdec_2025_scratch_both_views_true_bottleneck_min500_release_l2_v1"
PINNED_COMMIT = "231f72914c3ed5bd738443069437d8be698bc4a4"
REPO_URL = "https://github.com/vdat-18/mvdec_fpmax.git"
REPO = Path("/kaggle/temp/mvdec_fpmax")
OUTPUT = Path("/kaggle/working/mvdec-s2-40seeds-reuters")
TRAJECTORY = Path("/kaggle/working/mvdec-s2-40seeds-reuters-trajectory")
BRANCH = "claude/mvdec-hypotheses-2026-10-24"

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


for path in (REPO, OUTPUT, TRAJECTORY):
    if path.exists():
        shutil.rmtree(path)
OUTPUT.mkdir(parents=True)
TRAJECTORY.mkdir(parents=True)
subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "checkout", "-q", PINNED_COMMIT], cwd=REPO, check=True)
TRAINING_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert TRAINING_COMMIT == PINNED_COMMIT, (TRAINING_COMMIT, PINNED_COMMIT)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Training commit:", TRAINING_COMMIT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "gdown"], check=True)
sys.path.insert(0, str(REPO / "src"))
subprocess.run(
    [sys.executable, str(REPO / "scripts/fetch_dekm_release.py"), "--datasets", "REUTERS"],
    cwd=REPO,
    check=True,
)

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_mvdec_independent_view2.py",
        "tests/test_mvdec_contract.py",
        "tests/test_data.py",
        "tests/test_mvdec_architecture.py",
        "tests/test_mvdec_trajectory.py",
    ],
    cwd=REPO,
    env={**os.environ, "PYTHONPATH": os.pathsep.join([str(REPO), str(REPO / "src")])},
    check=True,
)


In [ ]:
for seed in range(52, 92):
    command = [
        sys.executable, "-u", "src/representation_learning/MVDEC_dense.py", "REUTERS",
        "--runs", "1", "--seed", str(seed),
        "--protocol", PROTOCOL,
        "--progress-interval", "100",
        "--dataset-root", "external_repos/DEKM/datasets",
        "--public-output-dir", str(OUTPUT),
    ]
    print("Running:", " ".join(command), flush=True)
    env = {**os.environ, "MVDEC_LABEL_TRAJECTORY_LOG": str(TRAJECTORY / f"seed_{seed}.jsonl")}
    subprocess.run(command, cwd=REPO, env=env, check=False, timeout=4 * 60 * 60)


In [ ]:
import statistics

PAPER = {'view1': (0.7642, 0.5999), 'view2': (0.6501, 0.5129), 'fused': (0.7813, 0.6141)}
manifests = sorted(OUTPUT.rglob("manifest.json"))
print("complete manifests:", len(manifests))
rows = []
for path in manifests:
    manifest = json.loads(path.read_text(encoding="utf-8"))
    assert manifest["status"] == "complete" and manifest["protocol_id"] == PROTOCOL
    rows.append((manifest["seed"], manifest["metrics"]))
rows.sort(key=lambda row: row[0])

print("seed  " + "  ".join(key + "_acc/nmi" for key in PAPER))
for seed, metrics in rows:
    print(seed, "  ".join("%.4f/%.4f" % (metrics[key + "_acc"], metrics[key + "_nmi"]) for key in PAPER))
print()
print("rep    mean_ACC    std  median   best | paper_ACC | mean_NMI best_NMI | paper_NMI")
for key, (paper_acc, paper_nmi) in PAPER.items():
    accs = [metrics[key + "_acc"] for _, metrics in rows]
    nmis = [metrics[key + "_nmi"] for _, metrics in rows]
    print("%-6s %8.4f %6.4f %7.4f %7.4f | %.4f | %.4f %.4f | %.4f" % (key, statistics.mean(accs), statistics.stdev(accs), statistics.median(accs), max(accs), paper_acc, statistics.mean(nmis), max(nmis), paper_nmi))

best = []
for path in sorted(TRAJECTORY.glob("seed_*.jsonl")):
    traj = [json.loads(line) for line in path.read_text().splitlines()]
    top = max(traj, key=lambda r: r["fused_acc"])
    best.append((path.stem, top["step"], top["fused_acc"], top["fused_nmi"], max(r["fused_nmi"] for r in traj),
                 max(r["view2_acc"] for r in traj), max(r["view2_nmi"] for r in traj)))
print("seed step_bestACC fused_acc fused_nmi | max_fused_nmi max_view2_acc max_view2_nmi")
for row in best:
    print("%s %d %.4f %.4f | %.4f %.4f %.4f" % row)
shutil.copytree(TRAJECTORY, OUTPUT / "trajectory")
zip_path = Path(shutil.make_archive("/kaggle/working/mvdec-s2-40seeds-reuters-results", "zip", OUTPUT.parent, OUTPUT.name))
digest = file_sha256(zip_path)
Path(str(zip_path) + ".sha256").write_text(digest + "  " + zip_path.name + chr(10), encoding="ascii")
print("ZIP:", zip_path)
print("SHA-256:", digest)


In [ ]:
for module, extra in (
    ("pipeline.mvdec_audit", ["--require-all-complete"]),
):
    subprocess.run(
        [
            sys.executable, "-m", module,
            "--root", str(OUTPUT),
            "--expected-datasets", "REUTERS",
            "--expected-protocol-id", PROTOCOL,
            "--expected-training-commit", TRAINING_COMMIT,
            *extra,
            *(["--dataset-root", "external_repos/DEKM/datasets"] if module.endswith("mvdec_audit") else []),
        ],
        cwd=REPO,
        check=True,
    )